## Store Sales - Time Series Forecasting


### Imports


In [1]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna

from optuna.integration.lightgbm import LightGBMPruningCallback
from optuna.trial import TrialState

warnings.filterwarnings("ignore")

### Config


In [2]:
SEED = 42
np.random.seed(SEED)

N_TRIALS = 30
MAX_BOOST_ROUNDS = 5000

EARLY_STOPPING_TUNE = 80
LOG_EVAL_PERIOD = 200

HORIZON_DAYS = 16  # competition horizon
NUM_THREADS = max(1, (os.cpu_count() or 4))

RUN_EDA_PRINTS = True  # set True for EDA

### Import Data


In [ ]:
# Paths
DATA_DIR = Path(r"C:/Users/Han/Projects/sstsf")  # enter path to datasets

# Read Data
datasets = {
    "train": pd.read_csv(
        DATA_DIR / "train.csv",
        usecols=["id", "date", "store_nbr", "family", "sales", "onpromotion"],
        dtype={
            "id": "uint64",
            "store_nbr": "category",
            "family": "category",
            "sales": "float32",
            "onpromotion": "uint32",
        },
        parse_dates=["date"],
    ),
    "stores": pd.read_csv(
        DATA_DIR / "stores.csv",
        dtype={
            "store_nbr": "category",
            "city": "category",
            "state": "category",
            "type": "category",
            "cluster": "category",
        },
    ),
    "holidays_events": pd.read_csv(
        DATA_DIR / "holidays_events.csv",
        usecols=["date", "type", "locale", "locale_name", "description", "transferred"],
        dtype={
            "type": "category",
            "locale": "category",
            "locale_name": "category",
            "description": "category",
            "transferred": "bool",
        },
        parse_dates=["date"],
    ),
    "oil": pd.read_csv(
        DATA_DIR / "oil.csv",
        dtype={"dcoilwtico": "float32"},
        parse_dates=["date"],
    ),
    "transactions": pd.read_csv(
        DATA_DIR / "transactions.csv",
        dtype={
            "store_nbr": "category",
            "transactions": "uint32",
        },
        parse_dates=["date"],
    ),
    "test": pd.read_csv(
        DATA_DIR / "test.csv",
        usecols=["id", "date", "store_nbr", "family", "onpromotion"],
        dtype={
            "id": "uint64",
            "store_nbr": "category",
            "family": "category",
            "onpromotion": "uint32",
        },
        parse_dates=["date"],
    ),
}

### EDA


In [4]:
if RUN_EDA_PRINTS:
    for name, df in datasets.items():
        print(f"\n{'='*15}\n{name}\n{'='*15}")
        print(df.head())
        print(df.info())

    for name, df in datasets.items():
        print(f"\n{'='*15}\n{name}\n{'='*15}")
        print("duplicates:", df.duplicated().sum())
        print("nulls:\n", df.isnull().sum())

    for name, df in datasets.items():
        print(f"\n{'='*15}\n{name}\n{'='*15}")
        print(df.describe().round())


train
   id       date store_nbr      family  sales  onpromotion
0   0 2013-01-01         1  AUTOMOTIVE    0.0            0
1   1 2013-01-01         1   BABY CARE    0.0            0
2   2 2013-01-01         1      BEAUTY    0.0            0
3   3 2013-01-01         1   BEVERAGES    0.0            0
4   4 2013-01-01         1       BOOKS    0.0            0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           uint64        
 1   date         datetime64[ns]
 2   store_nbr    category      
 3   family       category      
 4   sales        float32       
 5   onpromotion  uint32        
dtypes: category(2), datetime64[ns](1), float32(1), uint32(1), uint64(1)
memory usage: 74.4 MB
None

stores
  store_nbr           city                           state type cluster
0         1          Quito                       Pichincha    D      13
1         2

### Data Cleaning


In [5]:
# Clean Oil (forward-filled)
oil = datasets["oil"].sort_values("date").copy()
oil["dcoilwtico"] = oil["dcoilwtico"].ffill()
oil["dcoilwtico"] = (
    oil["dcoilwtico"].fillna(-1.0).astype("float32")
)  # leading NaNs only

# References
train = datasets["train"]
test = datasets["test"]
stores = datasets["stores"]
hol = datasets["holidays_events"]
transactions = datasets["transactions"]

### Feature Engineering


In [6]:
# Mark train/test then concat
train = train.copy()
test = test.copy()

train["is_train_flag"] = 1
test["is_train_flag"] = 0
test["sales"] = np.nan

all_df = pd.concat([train, test], ignore_index=True, sort=False)

# Ensure consistent dtypes
all_df["store_nbr"] = all_df["store_nbr"].astype("category")
all_df["family"] = all_df["family"].astype("category")

# Promotions
all_df["onpromotion"] = all_df["onpromotion"].fillna(0).astype("float32")

# Sales: only for lag/rolling computations
all_df["sales"] = all_df["sales"].astype("float32")
all_df["sales_filled"] = all_df["sales"].fillna(0.0).astype("float32")

# Precompute ranges/values
store_vals = all_df["store_nbr"].cat.categories
family_vals = all_df["family"].cat.categories
min_date = all_df["date"].min()
max_date = all_df["date"].max()
date_range = pd.date_range(min_date, max_date, freq="D")

In [7]:
# Store metadata
all_df = all_df.merge(stores, on="store_nbr", how="left")
for c in ["city", "state", "type", "cluster"]:
    all_df[c] = all_df[c].astype("category")

In [8]:
# Holiday features
hol = hol.copy()
hol = hol.loc[hol["transferred"] == False].drop(columns=["transferred"])

hol["is_holiday"] = (
    hol["type"]
    .isin(["Holiday", "Additional", "Bridge", "Event", "Transfer"])
    .astype("int8")
)
hol["is_workday"] = (hol["type"] == "Work Day").astype("int8")

hol_nat = (
    hol.loc[hol["locale"] == "National"]
    .groupby("date", observed=True)
    .agg(
        holiday_nat=("is_holiday", "max"),
        workday_nat=("is_workday", "max"),
        nat_type=("type", "first"),
        nat_desc=("description", "first"),
    )
    .reset_index()
)

hol_reg = (
    hol.loc[hol["locale"] == "Regional"]
    .groupby(["date", "locale_name"], observed=True)
    .agg(
        holiday_reg=("is_holiday", "max"),
        workday_reg=("is_workday", "max"),
        reg_type=("type", "first"),
        reg_desc=("description", "first"),
    )
    .reset_index()
    .rename(columns={"locale_name": "state"})
)

hol_loc = (
    hol.loc[hol["locale"] == "Local"]
    .groupby(["date", "locale_name"], observed=True)
    .agg(
        holiday_loc=("is_holiday", "max"),
        workday_loc=("is_workday", "max"),
        loc_type=("type", "first"),
        loc_desc=("description", "first"),
    )
    .reset_index()
    .rename(columns={"locale_name": "city"})
)

# Align merge key dtypes to reduce surprises
hol_reg["state"] = hol_reg["state"].astype(all_df["state"].dtype)
hol_loc["city"] = hol_loc["city"].astype(all_df["city"].dtype)

all_df = all_df.merge(hol_nat, on="date", how="left")
all_df = all_df.merge(hol_reg, on=["date", "state"], how="left")
all_df = all_df.merge(hol_loc, on=["date", "city"], how="left")

for c in [
    "holiday_nat",
    "workday_nat",
    "holiday_reg",
    "workday_reg",
    "holiday_loc",
    "workday_loc",
]:
    all_df[c] = all_df[c].fillna(0).astype("int8")

all_df["is_holiday_any"] = (
    all_df[["holiday_nat", "holiday_reg", "holiday_loc"]].max(axis=1).astype("int8")
)
all_df["is_workday_any"] = (
    all_df[["workday_nat", "workday_reg", "workday_loc"]].max(axis=1).astype("int8")
)

for c in ["nat_type", "nat_desc", "reg_type", "reg_desc", "loc_type", "loc_desc"]:
    all_df[c] = all_df[c].astype("category")

In [9]:
# Oil features
oil = oil.sort_values("date").copy()
oil["oil_lag_1"] = oil["dcoilwtico"].shift(1)
oil["oil_lag_7"] = oil["dcoilwtico"].shift(7)
oil["oil_lag_14"] = oil["dcoilwtico"].shift(14)

oil["oil_roll_mean_7"] = oil["dcoilwtico"].shift(1).rolling(7, min_periods=1).mean()
oil["oil_roll_mean_28"] = oil["dcoilwtico"].shift(1).rolling(28, min_periods=1).mean()
oil["oil_roll_std_28"] = oil["dcoilwtico"].shift(1).rolling(28, min_periods=2).std()

all_df = all_df.merge(
    oil[
        [
            "date",
            "dcoilwtico",
            "oil_lag_1",
            "oil_lag_7",
            "oil_lag_14",
            "oil_roll_mean_7",
            "oil_roll_mean_28",
            "oil_roll_std_28",
        ]
    ],
    on="date",
    how="left",
)

In [10]:
# Transaction features
transactions = transactions.copy()
transactions["transactions"] = transactions["transactions"].astype("float32")
transactions["store_nbr"] = transactions["store_nbr"].astype(all_df["store_nbr"].dtype)

txn_index = pd.MultiIndex.from_product(
    [store_vals, date_range], names=["store_nbr", "date"]
)
transactions = (
    transactions.set_index(["store_nbr", "date"]).sort_index().reindex(txn_index)
)

transactions["txn_was_missing"] = transactions["transactions"].isna().astype("int8")
transactions["transactions"] = (
    transactions["transactions"].fillna(0.0).astype("float32")
)

transactions = transactions.reset_index()
all_df = all_df.merge(transactions, on=["store_nbr", "date"], how="left")
all_df["transactions"] = all_df["transactions"].fillna(0.0).astype("float32")
all_df["txn_was_missing"] = all_df["txn_was_missing"].fillna(1).astype("int8")

In [11]:
# Date/time features
d = all_df["date"]
all_df["year"] = d.dt.year.astype("int16")
all_df["month"] = d.dt.month.astype("int8")
all_df["day"] = d.dt.day.astype("int8")
all_df["dayofweek"] = d.dt.dayofweek.astype("int8")
all_df["dayofyear"] = d.dt.dayofyear.astype("int16")
all_df["weekofyear"] = d.dt.isocalendar().week.astype("int16")

all_df["is_weekend"] = (all_df["dayofweek"] >= 5).astype("int8")
all_df["is_month_start"] = d.dt.is_month_start.astype("int8")
all_df["is_month_end"] = d.dt.is_month_end.astype("int8")
all_df["is_quarter_start"] = d.dt.is_quarter_start.astype("int8")
all_df["is_quarter_end"] = d.dt.is_quarter_end.astype("int8")

all_df["is_payday_15"] = (all_df["day"] == 15).astype("int8")
all_df["is_payday_eom"] = all_df["is_month_end"].astype("int8")

all_df["time_idx"] = (d - d.min()).dt.days.astype("int32")

# Cyclical encodings
all_df["dow_sin"] = np.sin(2 * np.pi * all_df["dayofweek"] / 7).astype("float32")
all_df["dow_cos"] = np.cos(2 * np.pi * all_df["dayofweek"] / 7).astype("float32")
all_df["doy_sin"] = np.sin(2 * np.pi * all_df["dayofyear"] / 365.25).astype("float32")
all_df["doy_cos"] = np.cos(2 * np.pi * all_df["dayofyear"] / 365.25).astype("float32")

In [12]:
# History-based features
all_df = all_df.sort_values(
    ["store_nbr", "family", "date"], kind="mergesort"
).reset_index(drop=True)

g = all_df.groupby(["store_nbr", "family"], sort=False, observed=True)

# Sales lags: only >= 16 so test horizon never uses missing sales
sales_lags = [16, 17, 21, 28, 35, 42, 56, 84, 112, 140, 168, 365]
for lag in sales_lags:
    all_df[f"sales_lag_{lag}"] = g["sales_filled"].shift(lag).astype("float32")

# Sales rolling anchored at t-16
sales_shift_16 = g["sales_filled"].shift(16)
for w in [7, 14, 28, 56]:
    all_df[f"sales_roll_mean_{w}_at16"] = (
        sales_shift_16.groupby([all_df["store_nbr"], all_df["family"]], sort=False)
        .rolling(w, min_periods=1)
        .mean()
        .reset_index(level=[0, 1], drop=True)
        .astype("float32")
    )
    all_df[f"sales_roll_std_{w}_at16"] = (
        sales_shift_16.groupby([all_df["store_nbr"], all_df["family"]], sort=False)
        .rolling(w, min_periods=2)
        .std()
        .reset_index(level=[0, 1], drop=True)
        .astype("float32")
    )

# Promotions: known for test
promo_lags = [1, 7, 14, 28]
for lag in promo_lags:
    all_df[f"promo_lag_{lag}"] = g["onpromotion"].shift(lag).astype("float32")

promo_shift_1 = g["onpromotion"].shift(1)
for w in [7, 14, 28]:
    all_df[f"promo_roll_sum_{w}"] = (
        promo_shift_1.groupby([all_df["store_nbr"], all_df["family"]], sort=False)
        .rolling(w, min_periods=1)
        .sum()
        .reset_index(level=[0, 1], drop=True)
        .astype("float32")
    )
    all_df[f"promo_roll_mean_{w}"] = (
        promo_shift_1.groupby([all_df["store_nbr"], all_df["family"]], sort=False)
        .rolling(w, min_periods=1)
        .mean()
        .reset_index(level=[0, 1], drop=True)
        .astype("float32")
    )

# Transactions: not provided for test, so only >=16 and anchored at t-16
tg = all_df.groupby("store_nbr", sort=False, observed=True)

txn_lags = [16, 17, 21, 28, 35, 42, 56, 84]
for lag in txn_lags:
    all_df[f"txn_lag_{lag}"] = tg["transactions"].shift(lag).astype("float32")

txn_shift_16 = tg["transactions"].shift(16)
for w in [7, 14, 28]:
    all_df[f"txn_roll_mean_{w}_at16"] = (
        txn_shift_16.groupby(all_df["store_nbr"], sort=False)
        .rolling(w, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
        .astype("float32")
    )
    all_df[f"txn_roll_std_{w}_at16"] = (
        txn_shift_16.groupby(all_df["store_nbr"], sort=False)
        .rolling(w, min_periods=2)
        .std()
        .reset_index(level=0, drop=True)
        .astype("float32")
    )

### Split back to train/test


In [13]:
train_fe = all_df.loc[all_df["is_train_flag"] == 1].copy()
test_fe = all_df.loc[all_df["is_train_flag"] == 0].copy()

y = train_fe["sales"].astype("float32")
test_ids = test_fe["id"].astype("uint64").copy()

drop_cols = [
    "sales",
    "sales_filled",
    "id",
    "date",
    "is_train_flag",
    # raw transactions not known in test; keep only lag/anchored features
    "transactions",
    "txn_was_missing",
]

features = [c for c in train_fe.columns if c not in drop_cols]

X = train_fe[features].copy()
X_test = test_fe[features].copy()

# Ensure object -> category
obj_cols = X.select_dtypes(include=["object"]).columns.tolist()
for c in obj_cols:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")

# Align category levels between train/test
cat_cols = X.select_dtypes(include=["category"]).columns.tolist()
for c in cat_cols:
    cats = pd.Index(X[c].cat.categories).union(pd.Index(X_test[c].cat.categories))
    X[c] = X[c].cat.set_categories(cats)
    X_test[c] = X_test[c].cat.set_categories(cats)

### Time-based Validation (last 16 days of TRAIN)


In [14]:
last_train_date = train_fe["date"].max()
val_start = last_train_date - pd.Timedelta(days=HORIZON_DAYS - 1)  # inclusive

tr_mask = train_fe["date"] < val_start
va_mask = train_fe["date"] >= val_start

X_tr = X.loc[tr_mask].copy()
X_va = X.loc[va_mask].copy()
y_tr = y.loc[tr_mask].copy()
y_va = y.loc[va_mask].copy()

y_tr_log = np.log1p(y_tr.astype("float32"))
y_va_log = np.log1p(y_va.astype("float32"))

cat_features = [c for c in features if str(X_tr[c].dtype) == "category"]

### Optuna Hyperparameter Tuning (LightGBM)


In [15]:
DATASET_PARAMS = {
    "max_bin": 255,
    "feature_pre_filter": False,
}

dtrain = lgb.Dataset(
    X_tr,
    label=y_tr_log,
    categorical_feature=cat_features,
    params=DATASET_PARAMS,
    free_raw_data=False,
)

dvalid = lgb.Dataset(
    X_va,
    label=y_va_log,
    categorical_feature=cat_features,
    reference=dtrain,
    params=DATASET_PARAMS,
    free_raw_data=False,
)

base_params = {
    "objective": "regression",
    "metric": "rmse",  # RMSE on log1p(sales) aligns with RMSLE
    "verbosity": -1,
    "seed": SEED,
    "force_col_wise": True,
    "num_threads": NUM_THREADS,
}

sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True)
pruner = optuna.pruners.MedianPruner(n_warmup_steps=200)
study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)

best_seen = np.inf

for t in range(N_TRIALS):
    trial = study.ask()

    # Depth / leaves (enforce num_leaves <= 2^max_depth - 1)
    max_depth = trial.suggest_int("max_depth", 6, 12)
    num_leaves_raw = trial.suggest_int("num_leaves", 32, 512, log=True)
    num_leaves = int(min(num_leaves_raw, (2**max_depth) - 1))
    num_leaves = max(num_leaves, 2)
    trial.set_user_attr("num_leaves_used", int(num_leaves))

    min_data_in_leaf = trial.suggest_int("min_data_in_leaf", 64, 2048, log=True)
    learning_rate = trial.suggest_float("learning_rate", 0.02, 0.10, log=True)

    feature_fraction = trial.suggest_float("feature_fraction", 0.6, 1.0)
    bagging_fraction = trial.suggest_float("bagging_fraction", 0.6, 1.0)
    bagging_freq = trial.suggest_int("bagging_freq", 1, 10)

    lambda_l1 = trial.suggest_float("lambda_l1", 0.0, 3.0)
    lambda_l2 = trial.suggest_float("lambda_l2", 0.0, 10.0)
    min_gain_to_split = trial.suggest_float("min_gain_to_split", 0.0, 0.5)

    extra_trees = trial.suggest_categorical("extra_trees", [False, True])

    params = {
        **base_params,
        "learning_rate": learning_rate,
        "max_depth": max_depth,
        "num_leaves": num_leaves,
        "min_data_in_leaf": min_data_in_leaf,
        "feature_fraction": feature_fraction,
        "bagging_fraction": bagging_fraction,
        "bagging_freq": bagging_freq,
        "lambda_l1": lambda_l1,
        "lambda_l2": lambda_l2,
        "min_gain_to_split": min_gain_to_split,
        "extra_trees": extra_trees,
    }

    try:
        model_t = lgb.train(
            params,
            dtrain,
            num_boost_round=MAX_BOOST_ROUNDS,
            valid_sets=[dvalid],
            valid_names=["valid"],
            callbacks=[
                lgb.early_stopping(stopping_rounds=EARLY_STOPPING_TUNE, verbose=False),
                LightGBMPruningCallback(trial, "rmse", valid_name="valid"),
            ],
        )

        score = float(model_t.best_score["valid"]["rmse"])
        trial.set_user_attr("best_iteration", int(model_t.best_iteration))

        study.tell(trial, score)

        if score < best_seen:
            best_seen = score
            print(
                f"[Trial {t+1}/{N_TRIALS}] New best RMSLE: {best_seen:.6f} | best_iter: {model_t.best_iteration}"
            )

    except optuna.TrialPruned:
        study.tell(trial, state=TrialState.PRUNED)
        continue

print("\nBest tuned RMSLE:", study.best_value)
print("Best tuned params:", study.best_trial.params)

best_iter = int(study.best_trial.user_attrs.get("best_iteration", 1000))
best_num_leaves = int(
    study.best_trial.user_attrs.get(
        "num_leaves_used", study.best_trial.params["num_leaves"]
    )
)
print("Best iteration:", best_iter)

[I 2026-01-29 22:54:18,241] A new study created in memory with name: no-name-7fd60da7-2b1c-4569-a6f6-b5295a908854


[Trial 1/30] New best RMSLE: 0.405800 | best_iter: 1113
[Trial 2/30] New best RMSLE: 0.400773 | best_iter: 3270
[Trial 3/30] New best RMSLE: 0.391467 | best_iter: 542
[Trial 15/30] New best RMSLE: 0.389136 | best_iter: 1315

Best tuned RMSLE: 0.38913607885471013
Best tuned params: {'max_depth': 8, 'num_leaves': 300, 'min_data_in_leaf': 119, 'learning_rate': 0.05455048481433844, 'feature_fraction': 0.6943921280727747, 'bagging_fraction': 0.7088496303069086, 'bagging_freq': 8, 'lambda_l1': 0.13260290631245092, 'lambda_l2': 3.71443993914097, 'min_gain_to_split': 0.4758456239937028, 'extra_trees': False}
Best iteration: 1315


### Model Training


In [16]:
best_params = {**base_params, **study.best_trial.params}
best_params["num_leaves"] = best_num_leaves

y_full_log = np.log1p(y.astype("float32"))

dfull = lgb.Dataset(
    X,
    label=y_full_log,
    categorical_feature=cat_features,
    params=DATASET_PARAMS,
    free_raw_data=False,
)

final_model = lgb.train(
    best_params,
    dfull,
    num_boost_round=best_iter,
    valid_sets=[dfull],
    valid_names=["train_full"],
    callbacks=[lgb.log_evaluation(period=LOG_EVAL_PERIOD)],
)

[200]	train_full's rmse: 0.405384
[400]	train_full's rmse: 0.385366
[600]	train_full's rmse: 0.374321
[800]	train_full's rmse: 0.368827
[1000]	train_full's rmse: 0.365865
[1200]	train_full's rmse: 0.364045


### Predict Test + Create Submission


In [17]:
test_pred_log = final_model.predict(
    X_test, num_iteration=final_model.current_iteration()
)
test_pred = np.expm1(test_pred_log).astype("float32")
test_pred = np.clip(test_pred, 0, None)

submission = pd.DataFrame({"id": test_ids.values, "sales": test_pred})
print(submission.head())

sub_path = DATA_DIR / "submission.csv"
submission.to_csv(sub_path, index=False)
print("Saved:", sub_path)

        id     sales
0  3000888  4.054369
1  3002670  3.574803
2  3004452  3.783364
3  3006234  3.673502
4  3008016  1.643920
Saved: C:\Users\Han\OneDrive\Documents\HW\Work\Projects\sstsf\submission.csv
